# Notebook 9: Results Figures
Figures for the manuscript that summarize notebooks 7 (citation counts) and 8 (FWCI) together, so they belong to neither of them:
- **Main figure:** split violins for *sharing* and *non-sharing* articles, citations on the left and FWCI on the right (each with its own y axis). Each panel is annotated with the sharing coefficient `b` and its p-value from Model 3, the venue-adjusted mixed-effects model reported in the Results.
- **Supplementary figure:** the mean and standard error of both outcomes for the three sharing granularities (PARTICIPANT, TRIAL, FIXATION), one panel above the other. There is no annotation, since there are two coefficients and two p-values.

Both outcomes are plotted on the scale the models are fitted on, `log(1 + citations)` and `log(imputed FWCI)`, so the annotated `b` matches the axis; the tick labels give the original units. `b` is unstandardized and on the log scale, so `exp(b) - 1` is the proportional difference.


In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from helpers.config import *
from helpers import dataset
from helpers.plotting import save_figure
from helpers.stats import sharing_effect

pio.renderers.default = "browser"

## Setup
### Helper Functions

In [2]:
def format_effect(effect: dict) -> str:
    p_text = f"<i>p={effect['p']:.3f}</i>" if effect["p"] >= 0.001 else "<i>p<0.001</i>"
    return f"<i>b={effect['b']:.2f}</i>, {p_text}"

## Panels
Both figures show the same two panels, citations and FWCI, each with its own y axis (the units differ). Tick labels are in the original units. Fonts follow the project scheme (`TITLE_FONT`, `AXIS_TITLE_FONT`, `AXIS_TICK_FONT`).


In [3]:
PANELS = [
    dict(
        outcome="log_citations", title="<b>Citations through 2025</b>", y_title="Citations (<i>log scale</i>)",
        ticks=[0, 2, 5, 10, 25, 50, 100], transform=np.log1p,
    ),
    dict(
        outcome="log_imputed_fwci", title="<b>Field-Weighted Citation Impact</b>", y_title="FWCI (<i>log scale</i>)",
        ticks=[0.1, 0.25, 0.5, 1, 2, 4], transform=np.log,
    ),
]


def style_panels(fig: go.Figure, positions: list[tuple[int, int]], ticks_by_outcome: dict[str, list[float]] | None = None) -> None:
    """Axis titles, original-unit tick labels and the subplot titles, in the project's font scheme."""
    fig.update_annotations(font=TITLE_FONT)         # the subplot titles (the only annotations so far)
    for panel, (row, col) in zip(PANELS, positions):
        ticks = (ticks_by_outcome or {}).get(panel["outcome"], panel["ticks"])
        fig.update_yaxes(
            row=row, col=col,
            title=dict(text=panel["y_title"], font=AXIS_TITLE_FONT, standoff=5),
            tickvals=panel["transform"](ticks), ticktext=[f"{tick:g}" for tick in ticks], tickfont=AXIS_TICK_FONT,
        )

### Load Data
The analytic sample is rebuilt (or loaded from the parquet cache) by `helpers.dataset.load_or_build()`, so this notebook runs standalone from a cold kernel. The two outcome frames are built as in notebooks 7 and 8.

In [4]:
combined, FEATURES_DF = dataset.load_or_build()

CITATIONS_DF = dataset.build_citations_frame(combined, FEATURES_DF)
FWCI_DF = dataset.to_smf_columns(
    pd.concat([FEATURES_DF, np.log(dataset.fwci_with_zeros_imputed(combined))], axis=1)
    .rename(columns={"fwci_with_zeros_imputed": "log(Imputed FWCI)"})
    .drop(columns=["log(Weeks Since Pub.)"])
)

OUTCOMES = pd.DataFrame({
    "is_sharing_data": CITATIONS_DF["is_sharing_data"], "sharing_class": CITATIONS_DF["sharing_class"],
    "log_citations": CITATIONS_DF["log_citations"], "log_imputed_fwci": FWCI_DF["log_imputed_fwci"],
})
OUTCOMES["is_sharing_data"].value_counts()

Loaded cached dataset from C:\Users\nirjo\Documents\University\PhD\Projects\eye_movement_data_sharing\data_store (234 rows)


is_sharing_data
0    154
1     80
Name: count, dtype: int64

## Model 3 Coefficients
The annotations come from Model 3 of notebooks 7 and 8: a mixed-effects model with venue as a random effect and the venue impact score as a covariate, fitted by REML.


In [5]:
SHARING_TERM = "C(is_sharing_data)[T.1]"

citations_model = smf.mixedlm(
    "log_citations ~ C(is_sharing_data) + C(has_us_author) + C(is_open_access) + C(has_preprint) + log_weeks_since_pub + log_number_of_authors + venue_impact",
    data=CITATIONS_DF,
    groups=CITATIONS_DF["venue_name"],
).fit(reml=True)
fwci_model = smf.mixedlm(
    "log_imputed_fwci ~ C(is_sharing_data) + C(has_us_author) + C(is_open_access) + C(has_preprint) + log_number_of_authors + venue_impact",
    data=FWCI_DF,
    groups=FWCI_DF["venue_name"],
).fit(reml=True)

EFFECTS = {
    "log_citations": sharing_effect(citations_model, SHARING_TERM),
    "log_imputed_fwci": sharing_effect(fwci_model, SHARING_TERM),
}

{outcome: format_effect(effect) for outcome, effect in EFFECTS.items()}

{'log_citations': '<i>b=0.04</i>, <i>p=0.774</i>',
 'log_imputed_fwci': '<i>b=0.07</i>, <i>p=0.666</i>'}

## Figure: Sharing vs. Non-Sharing


In [10]:
group_sizes = OUTCOMES["is_sharing_data"].value_counts()
VIOLIN_POSITIONS = [(1, 1), (1, 2)]

violin_fig = make_subplots(
    rows=1, cols=2, shared_yaxes=False, horizontal_spacing=0.14, subplot_titles=[panel["title"] for panel in PANELS],
)
style_panels(violin_fig, VIOLIN_POSITIONS)
violin_fig.update_annotations(yshift=45)            # lift the subplot titles above the coefficient

for panel, (row, col) in zip(PANELS, VIOLIN_POSITIONS):
    for is_share, label in ((0, "NOT SHARING"), (1, "SHARING")):
        values = OUTCOMES.loc[OUTCOMES["is_sharing_data"] == is_share, panel["outcome"]]
        color = CLASS_COLORS[label]
        violin_fig.add_trace(go.Violin(
            x=[""] * len(values), y=values,
            name=label.title(), legendgroup=label, scalegroup=panel["outcome"], showlegend=False,
            side="positive" if is_share else "negative",
            fillcolor=color, line_color=color, width=0.9, spanmode="hard", points=False,
            box=dict(visible=False), meanline=dict(visible=True, color="black", width=2),
        ), row=row, col=col)
        # group label and n below the half-violin
        violin_fig.add_annotation(
            row=row, col=col,
            text=f"{label.title()}",
            font=AXIS_TITLE_FONT,
            xref="x domain", x=0.73 if is_share else 0.27, xanchor="center",
            yref="y domain", y=0.0, yanchor="top", yshift=-8,
            showarrow=False,
        )
    # the sharing coefficient (Model 3), centred above the panel
    violin_fig.add_annotation(
        row=row, col=col, text=format_effect(EFFECTS[panel["outcome"]]), showarrow=False, font=AXIS_TITLE_FONT,
        xref="x domain", x=0.5, xanchor="center", yref="y domain", y=1.0, yanchor="top", yshift=10,
    )

violin_fig.update_xaxes(showticklabels=False, ticks="", zeroline=False)
violin_fig.update_layout(
    width=900, height=520,
    violinmode="overlay", violingap=0,
    legend=dict(visible=False),
    margin=dict(t=100, b=80, l=10, r=10),
    template="simple_white",
)

if False:
    violin_fig.show()

In [7]:
# flip to True to export this figure into `output/`
if False:
    save_figure(violin_fig, "results_sharing_violins.png", width=900, height=520)

## Supplementary Figure: Sharing Granularity
Mean and standard error (SEM) of each outcome on the plotted (log) scale, for the 80 sharing articles by granularity. The panels are stacked and each keeps its own x axis. The bars start at 0, which is zero citations in the top panel and FWCI = 1 (the field average) in the bottom one.


In [16]:
GRANULARITY_ORDER = [cls for cls in SHARING_CLASS_ORDER if cls != "NONE"]      # the three sharing classes

SHARERS = OUTCOMES.loc[OUTCOMES["sharing_class"].isin(GRANULARITY_ORDER)]
class_sizes = SHARERS["sharing_class"].value_counts()
class_labels = [f"{cls.title()}" for cls in GRANULARITY_ORDER]
BAR_POSITIONS = [(1, 1), (2, 1)]

bar_fig = make_subplots(
    rows=2, cols=1, shared_xaxes=False,
    vertical_spacing=0.125,
    subplot_titles=[panel["title"] for panel in PANELS],
)
# the group means span a narrower range than the violins, so the bar panels get their own ticks
style_panels(
    bar_fig,
    BAR_POSITIONS,
    ticks_by_outcome={"log_citations": [0, 2, 5, 10, 20, 30],"log_imputed_fwci": [0.5, 0.75, 1, 1.5, 2]}
)

for panel, (row, col) in zip(PANELS, BAR_POSITIONS):
    summary = SHARERS.groupby("sharing_class", observed=True)[panel["outcome"]].agg(["mean", "sem"]).loc[GRANULARITY_ORDER]
    bar_fig.add_trace(go.Bar(
        x=class_labels, y=summary["mean"],
        error_y=dict(type="data", array=summary["sem"], color="black", thickness=1.5, width=6),
        marker=dict(color=[CLASS_COLORS[cls] for cls in GRANULARITY_ORDER]), showlegend=False,
    ), row=row, col=col)

bar_fig.add_hline(
    row=2, col=1,
    y=0, line_dash="dash", line=dict(color="black", width=2), opacity=1.0,
    annotation=dict(
        text="<i>Field average</i>", font=AXIS_TICK_FONT,
        xref="paper", x=1.0, xanchor="right", xshift=5,
        yref="y", y=0, yanchor="top", yshift=-5,
        showarrow=False,
    ),
)
bar_fig.update_xaxes(tickfont=AXIS_TICK_FONT)
bar_fig.update_layout(
    width=600, height=800,
    showlegend=False,
    margin=dict(t=30, b=10, l=10, r=10),
    template="simple_white",
)

if False:
    bar_fig.show()

In [17]:
# flip to True to export this figure into `output/`
if False:
    save_figure(bar_fig, "supp_granularity_bars.png", width=600, height=800)

Saved figure to C:\Users\nirjo\Documents\University\PhD\Projects\eye_movement_data_sharing\output\supp_granularity_bars.png
